#  Bangalore Traffic Analysis
## Notebook 1 — Data Collection

This notebook fetches data from all sources:
- **OSMnx** – Road network
- **TomTom API** – Traffic congestion
- **Open-Meteo** – Weather & rainfall
- **BMTC GTFS (DataMeet)** – Bus stops & routes
- **data.gov.in / OpenCity** – Accidents

###  Install Dependencies
Run this cell once to install all required libraries.

In [1]:
# Run once
import subprocess, sys
pkgs = ["osmnx", "requests", "pandas", "sqlalchemy", "psycopg2-binary",
        "folium", "matplotlib", "seaborn", "tqdm"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs)
print("All packages installed.")


Defaulting to user installation because normal site-packages is not writeable
All packages installed.


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


###  Configuration
Fill in your API keys and database credentials here.

In [2]:
# ── API Keys ──────────────────────────────────────────────────────────────
TOMTOM_API_KEY = "F02Rqymm7NnxHDKTPHbAtOua1Uj0Y1rF"   # https://developer.tomtom.com

# ── PostgreSQL Connection ─────────────────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "database": "bangalore_traffic",
    "user":     "postgres",
    "password": "prabhupada",
}

# ── Bounding Box for Bangalore ────────────────────────────────────────────
BBOX = {
    "north": 13.1407,
    "south": 12.8340,
    "east":  77.7480,
    "west":  77.4600,
}
CITY_NAME = "Bangalore, Karnataka, India"

# ── Key Junctions to monitor (lat, lon, label) ────────────────────────────
JUNCTIONS = [
    (12.9716, 77.5946, "Silk Board"),
    (12.9352, 77.6245, "Marathahalli"),
    (12.9698, 77.7499, "Whitefield"),
    (12.9784, 77.6408, "Outer Ring Road KR Puram"),
    (12.9762, 77.6033, "Koramangala"),
    (12.9719, 77.5937, "HSR Layout"),
    (12.9856, 77.5533, "MG Road"),
    (13.0358, 77.5970, "Hebbal"),
    (12.9141, 77.6101, "BTM Layout"),
    (12.9902, 77.7172, "Mahadevapura"),
]

print("Config loaded.")
print(f"   Monitoring {len(JUNCTIONS)} junctions.")


Config loaded.
   Monitoring 10 junctions.


###  Setup PostgreSQL Database
Creates all tables needed for this project.

In [3]:
from sqlalchemy import create_engine, text

DB_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine = create_engine(DB_URL)

try:
    with engine.connect() as conn:
        print("Successfully connected to PostgreSQL database!")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Please ensure PostgreSQL is running and the database exists.")


SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS road_network (
    osmid        BIGINT PRIMARY KEY,
    name         TEXT,
    highway      TEXT,
    length_m     FLOAT,
    lanes        TEXT,
    maxspeed     TEXT,
    geometry_wkt TEXT
);

CREATE TABLE IF NOT EXISTS traffic_speeds (
    id              SERIAL PRIMARY KEY,
    junction_label  TEXT,
    latitude        FLOAT,
    longitude       FLOAT,
    fetched_at      TIMESTAMP,
    current_speed   FLOAT,
    free_flow_speed FLOAT,
    current_travel_time INT,
    free_flow_travel_time INT,
    confidence      FLOAT,
    road_closure    BOOLEAN
);

CREATE TABLE IF NOT EXISTS weather_data (
    id              SERIAL PRIMARY KEY,
    date            DATE,
    hour            INT,
    temperature_c   FLOAT,
    precipitation_mm FLOAT,
    windspeed_kmh   FLOAT,
    weathercode     INT
);

CREATE TABLE IF NOT EXISTS accidents (
    id         SERIAL PRIMARY KEY,
    date       DATE,
    time       TEXT,
    location   TEXT,
    latitude   FLOAT,
    longitude  FLOAT,
    severity   TEXT,
    vehicles   INT,
    casualties INT
);

CREATE TABLE IF NOT EXISTS bmtc_stops (
    stop_id    TEXT PRIMARY KEY,
    stop_name  TEXT,
    latitude   FLOAT,
    longitude  FLOAT
);

CREATE TABLE IF NOT EXISTS bmtc_routes (
    route_id    TEXT,
    stop_id     TEXT,
    stop_seq    INT,
    arrival_time TEXT
);
"""

with engine.connect() as conn:
    for stmt in SCHEMA_SQL.strip().split(";"):
        stmt = stmt.strip()
        if stmt:
            conn.execute(text(stmt))
    conn.commit()

print("All tables created (or already exist).")


Successfully connected to PostgreSQL database!
All tables created (or already exist).


### 1 OSMnx — Road Network

In [4]:
import osmnx as ox
import pandas as pd
from sqlalchemy import text

print("Downloading road network for Bangalore…")
G = ox.graph_from_place(CITY_NAME, network_type="drive")
_, edges = ox.graph_to_gdfs(G)

edges_df = edges.reset_index()[["osmid", "name", "highway", "length", "lanes", "maxspeed", "geometry"]]
edges_df.rename(columns={"length": "length_m"}, inplace=True)
print("Converting geometry format... (this takes just a moment)")
edges_df["geometry_wkt"] = edges_df["geometry"].to_wkt()
edges_df.drop(columns=["geometry"], inplace=True)
edges_df["osmid"] = edges_df["osmid"].apply(lambda x: x[0] if isinstance(x, list) else x)
edges_df.drop_duplicates(subset="osmid", inplace=True)

edges_df.to_sql("road_network", engine, if_exists="replace", index=False)
print(f"Road network saved: {len(edges_df):,} road segments.")
edges_df.head(10)


/Users/rishabhsharma/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Converting geometry format... (this takes just a moment)
Road network saved: 93,161 road segments.


,osmid,name,highway,length_m,lanes,maxspeed,geometry_wkt
0,32261256,2nd Main Road,residential,244.091315,2,NaN,"LINESTRING (77.598721 12.910542, 77.598478 12...."
1,1367650597,9th Cross Road,secondary,29.723619,2,NaN,"LINESTRING (77.598721 12.910542, 77.598994 12...."
2,1367650758,2nd Main Road,residential,5.961520,NaN,NaN,"LINESTRING (77.598721 12.910542, 77.598726 12...."
3,111814615,Mahayogi Vemana Road,primary,44.630750,2,NaN,"LINESTRING (77.624072 12.934965, 77.624371 12...."
4,28186701,7th Cross Road,residential,50.303992,NaN,NaN,"LINESTRING (77.624072 12.934965, 77.62402 12.9..."
5,8656177,3rd Main Road,residential,100.510660,NaN,NaN,"LINESTRING (77.629168 12.938493, 77.628877 12...."
6,545604697,Inner Ring Road,primary,56.390557,2,NaN,"LINESTRING (77.629168 12.938493, 77.629377 12...."
7,22838606,Yelahanka Road,primary,16.541215,4,80,"LINESTRING (77.594178 13.095224, 77.594068 13...."
8,1077664785,NaN,secondary,14.844541,NaN,NaN,"LINESTRING (77.594178 13.095224, 77.594286 13...."
9,45443685,NaN,residential,122.157793,NaN,NaN,"LINESTRING (77.579325 12.985951, 77.578405 12...."


In [5]:
edges_df.head(10)

,osmid,name,highway,length_m,lanes,maxspeed,geometry_wkt
0,32261256,2nd Main Road,residential,244.091315,2,NaN,"LINESTRING (77.598721 12.910542, 77.598478 12...."
1,1367650597,9th Cross Road,secondary,29.723619,2,NaN,"LINESTRING (77.598721 12.910542, 77.598994 12...."
2,1367650758,2nd Main Road,residential,5.961520,NaN,NaN,"LINESTRING (77.598721 12.910542, 77.598726 12...."
3,111814615,Mahayogi Vemana Road,primary,44.630750,2,NaN,"LINESTRING (77.624072 12.934965, 77.624371 12...."
4,28186701,7th Cross Road,residential,50.303992,NaN,NaN,"LINESTRING (77.624072 12.934965, 77.62402 12.9..."
5,8656177,3rd Main Road,residential,100.510660,NaN,NaN,"LINESTRING (77.629168 12.938493, 77.628877 12...."
6,545604697,Inner Ring Road,primary,56.390557,2,NaN,"LINESTRING (77.629168 12.938493, 77.629377 12...."
7,22838606,Yelahanka Road,primary,16.541215,4,80,"LINESTRING (77.594178 13.095224, 77.594068 13...."
8,1077664785,NaN,secondary,14.844541,NaN,NaN,"LINESTRING (77.594178 13.095224, 77.594286 13...."
9,45443685,NaN,residential,122.157793,NaN,NaN,"LINESTRING (77.579325 12.985951, 77.578405 12...."


### 2 TomTom API — Real-Time Traffic Speeds

> **Note:** Free tier allows ~2,500 calls/day. This cell fetches live data for the configured junctions and appends to the `traffic_speeds` table. Schedule it (e.g., via cron or Task Scheduler) to build a historical dataset.

In [6]:
import requests
from datetime import datetime, timedelta
import pandas as pd
from datetime import datetime

BASE_URL = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"

records = []
for lat, lon, label in JUNCTIONS:
    params = {
        "key":   TOMTOM_API_KEY,
        "point": f"{lat},{lon}",
        "unit":  "KMPH",
    }
    try:
        r = requests.get(BASE_URL, params=params, timeout=10)
        r.raise_for_status()
        d = r.json()["flowSegmentData"]
        records.append({
            "junction_label":         label,
            "latitude":               lat,
            "longitude":              lon,
            "fetched_at":             datetime.now(),
            "current_speed":          d.get("currentSpeed"),
            "free_flow_speed":        d.get("freeFlowSpeed"),
            "current_travel_time":    d.get("currentTravelTime"),
            "free_flow_travel_time":  d.get("freeFlowTravelTime"),
            "confidence":             d.get("confidence"),
            "road_closure":           d.get("roadClosure", False),
        })
        print(f"  {label}: {d.get('currentSpeed')} km/h  (free-flow {d.get('freeFlowSpeed')} km/h)")
    except Exception as e:
        print(f"  WARNING: {label}: {e}")

if records:
    df = pd.DataFrame(records)
    df.to_sql("traffic_speeds", engine, if_exists="append", index=False)
    print(f"\n{len(records)} junction records saved to traffic_speeds.")
else:
    print("No records fetched. Check your API key.")


  Silk Board: 19 km/h  (free-flow 26 km/h)
  Marathahalli: 25 km/h  (free-flow 31 km/h)
  Whitefield: 28 km/h  (free-flow 38 km/h)
  Outer Ring Road KR Puram: 11 km/h  (free-flow 22 km/h)
  Koramangala: 19 km/h  (free-flow 26 km/h)
  HSR Layout: 30 km/h  (free-flow 30 km/h)
  MG Road: 25 km/h  (free-flow 34 km/h)
  Hebbal: 32 km/h  (free-flow 47 km/h)
  BTM Layout: 27 km/h  (free-flow 27 km/h)
  Mahadevapura: 21 km/h  (free-flow 30 km/h)

10 junction records saved to traffic_speeds.


### 3 Open-Meteo — Historical Weather & Rainfall

Fetches hourly weather data for Bangalore for the past 365 days (free, no key required).

In [7]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":        12.9716,
    "longitude":       77.5946,
    "start_date":      "2024-01-01",
    "end_date":        "2024-12-31",
    "hourly":          "temperature_2m,precipitation,windspeed_10m,weathercode",
    "timezone":        "Asia/Kolkata",
}

print("Fetching weather data from Open-Meteo…")
r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
data = r.json()["hourly"]

weather_df = pd.DataFrame({
    "datetime":          pd.to_datetime(data["time"]),
    "temperature_c":     data["temperature_2m"],
    "precipitation_mm":  data["precipitation"],
    "windspeed_kmh":     data["windspeed_10m"],
    "weathercode":       data["weathercode"],
})
weather_df["date"] = weather_df["datetime"].dt.date
weather_df["hour"] = weather_df["datetime"].dt.hour
weather_df.drop(columns=["datetime"], inplace=True)

weather_df.to_sql("weather_data", engine, if_exists="replace", index=False)
print(f"Weather data saved: {len(weather_df):,} hourly records.")
weather_df.head(3)


Fetching weather data from Open-Meteo…
Weather data saved: 8,784 hourly records.


,temperature_c,precipitation_mm,windspeed_kmh,weathercode,date,hour
0,17.6,0.0,14.2,0,2024-01-01,0
1,16.9,0.0,14.4,1,2024-01-01,1
2,16.6,0.0,14.4,3,2024-01-01,2


In [8]:
weather_df.head(10)

,temperature_c,precipitation_mm,windspeed_kmh,weathercode,date,hour
0,17.6,0.0,14.2,0,2024-01-01,0
1,16.9,0.0,14.4,1,2024-01-01,1
2,16.6,0.0,14.4,3,2024-01-01,2
3,16.9,0.0,15.1,3,2024-01-01,3
4,17.2,0.0,14.4,3,2024-01-01,4
5,17.2,0.0,12.0,3,2024-01-01,5
6,16.9,0.0,9.7,3,2024-01-01,6
7,17.4,0.0,13.0,3,2024-01-01,7
8,19.5,0.0,13.7,3,2024-01-01,8
9,21.9,0.0,12.4,2,2024-01-01,9


### 4 BMTC GTFS — Bus Stops & Routes

Downloads BMTC GTFS data from DataMeet. If the URL changes, download the zip manually from [DataMeet](https://github.com/datameet/bmtc) and place `stops.txt` and `stop_times.txt` in the working directory.

In [9]:
import requests, zipfile, io, os
import pandas as pd

GTFS_URL = "https://github.com/datameet/bmtc/raw/master/data/bmtc_gtfs.zip"

print("Downloading BMTC GTFS data…")
try:
    r = requests.get(GTFS_URL, timeout=60)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall("bmtc_gtfs/")
    print("GTFS zip extracted to bmtc_gtfs/")
except Exception as e:
    print(f"WARNING: Auto-download failed ({e})")
    print("   Please manually download the GTFS zip from:")
    print("   https://github.com/datameet/bmtc")
    print("   and extract it into a folder called bmtc_gtfs/")

# Load stops
stops_path = "bmtc_gtfs/stops.txt"
if os.path.exists(stops_path):
    stops_df = pd.read_csv(stops_path, usecols=["stop_id", "stop_name", "stop_lat", "stop_lon"])
    stops_df.rename(columns={"stop_lat": "latitude", "stop_lon": "longitude"}, inplace=True)
    stops_df.to_sql("bmtc_stops", engine, if_exists="replace", index=False)
    print(f"BMTC stops saved: {len(stops_df):,} stops.")
    stops_df.head(3)
else:
    print("stops.txt not found — skipping.")


   Please manually download the GTFS zip from:
   https://github.com/datameet/bmtc
   and extract it into a folder called bmtc_gtfs/
BMTC stops saved: 10 stops.


In [10]:
# Load stop_times (routes)
import os, pandas as pd

st_path = "bmtc_gtfs/stop_times.txt"
if os.path.exists(st_path):
    routes_df = pd.read_csv(st_path, usecols=["trip_id", "stop_id", "stop_sequence", "arrival_time"])
    routes_df.rename(columns={"trip_id": "route_id", "stop_sequence": "stop_seq",
                               "arrival_time": "arrival_time"}, inplace=True)
    routes_df.to_sql("bmtc_routes", engine, if_exists="replace", index=False)
    print(f"BMTC routes saved: {len(routes_df):,} stop-time records.")
else:
    print("stop_times.txt not found — skipping.")


BMTC routes saved: 7 stop-time records.


### 5 Accident Data — data.gov.in / OpenCity

The accident CSV must be manually downloaded from [data.gov.in](https://data.gov.in) or [OpenCity](https://opencity.in). Search for **'Bangalore road accidents'**. Once downloaded, update the path below.

In [11]:
import pandas as pd, os

ACCIDENT_CSV = "accidents_bangalore.csv"   # ← update path if needed

if os.path.exists(ACCIDENT_CSV):
    acc_df = pd.read_csv(ACCIDENT_CSV)
    print("Columns found:", acc_df.columns.tolist())
    
    # Common column mapping — adjust based on actual file
    col_map = {
        "Date":       "date",
        "Time":       "time",
        "Location":   "location",
        "Latitude":   "latitude",
        "Longitude":  "longitude",
        "Severity":   "severity",
        "Vehicles":   "vehicles",
        "Casualties": "casualties",
    }
    acc_df.rename(columns={k: v for k, v in col_map.items() if k in acc_df.columns}, inplace=True)
    acc_df.to_sql("accidents", engine, if_exists="replace", index=False)
    print(f"Accident data saved: {len(acc_df):,} records.")
    acc_df.head(3)
else:
    print(f"WARNING: File '{ACCIDENT_CSV}' not found.")
    print("   Download from https://data.gov.in — search 'Bangalore road accidents'")
    print("   Place the CSV in the same folder as this notebook.")


Columns found: ['date', 'time', 'location', 'latitude', 'longitude', 'severity', 'vehicles', 'casualties', 'violation_type', 'fatalities', 'injuries']
Accident data saved: 10 records.


### Collection Summary

In [7]:
with engine.connect() as conn:
    tables = ["road_network", "traffic_speeds", "weather_data", "accidents", "bmtc_stops", "bmtc_routes"]
    print("=== Database Row Counts ===")
    for t in tables:
        try:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
            print(f"  {t:<25} {count:>8,} rows")
        except Exception as e:
            print(f"  {t:<25} ERROR: {e}")


NameError: name 'engine' is not defined